# LifeFile MongoDB connection in Google Colab

This notebook connects Colab to the LifeFile `scos` database hosted in MongoDB Atlas.

## One-time setup on Windows

Your local database is normally `mongodb://localhost:27017/scos`. Colab cannot access that local address directly. First create an Atlas cluster and database user, then copy the local database to Atlas from a Windows terminal:

```powershell
mongodump --uri="mongodb://localhost:27017/scos" --out="E:\lifefile-backup"
mongorestore --uri="mongodb+srv://<user>:<password>@<atlas-host>/scos" --drop "E:\lifefile-backup\scos"
```

Replace the placeholders locally. Never commit the Atlas URI or put it in a notebook cell.

In Colab, open **Secrets** (key icon), create a secret named `MONGO_URI`, paste the Atlas URI there, and enable notebook access.

In [ ]:
!pip -q install pymongo pandas

In [ ]:
import os
from google.colab import userdata
from pymongo import MongoClient

MONGO_URI = userdata.get("MONGO_URI")
if not MONGO_URI:
    raise RuntimeError("Create a Colab Secret named MONGO_URI before running this cell.")

client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10000)
client.admin.command("ping")
db = client["scos"]

print("Connected to LifeFile MongoDB Atlas")
print("Collections:", sorted(db.list_collection_names()))

In [ ]:
import pandas as pd

collection_names = ["appointments", "doctors", "hospitals", "reviews"]
frames = {}

for name in collection_names:
    frames[name] = pd.DataFrame(list(db[name].find({}, {"_id": 0})))
    print(f"{name}: {len(frames[name]):,} rows")

appointments = frames["appointments"]
doctors = frames["doctors"]
hospitals = frames["hospitals"]
reviews = frames["reviews"]

In [ ]:
display(appointments.head())
display(doctors.head())
display(hospitals.head())
display(reviews.head())

## Optional CSV export

Run this cell to save the connected collections for offline cleaning or sharing with the BI lab.

In [ ]:
from google.colab import files

for name, frame in frames.items():
    path = f"/content/{name}.csv"
    frame.to_csv(path, index=False)
    print(path)

# Download a selected export when needed:
# files.download("/content/appointments.csv")